<div style="background: linear-gradient(135deg, #0dcaf0 0%, #0d6efd 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🎲 03 — Bayes e Entropia</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 01 · Bloco 07 · Probabilidade</p>
</div>

## 🎯 Objetivos deste notebook

Ao finalizar este notebook você vai:
- Entender o Teorema de Bayes e aplicá-lo a problemas reais
- Saber o que é Entropia e como ela mede incerteza
- Implementar Cross-Entropy (a loss function mais usada em classificação)
- Conectar esses conceitos com Árvores de Decisão e Redes Neurais

---

## 1️⃣ Probabilidade Condicional

`P(A|B)` = probabilidade de A **dado que** B já aconteceu.

**Exemplo médico:**
- P(Doença) = 1% → probabilidade a priori
- P(Teste+|Doença) = 99% → sensibilidade
- P(Teste+|Saudável) = 5% → falso positivo

Se o teste deu positivo, qual a probabilidade real de ter a doença?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Teorema de Bayes: P(A|B) = P(B|A) · P(A) / P(B)

# Dados
P_doenca = 0.01          # 1% da população tem a doença
P_saudavel = 1 - P_doenca
P_positivo_dado_doenca = 0.99    # sensibilidade
P_positivo_dado_saudavel = 0.05  # taxa de falso positivo

# P(Teste+) = P(+|D)·P(D) + P(+|S)·P(S)  [lei da probabilidade total]
P_positivo = (P_positivo_dado_doenca * P_doenca + 
              P_positivo_dado_saudavel * P_saudavel)

# Bayes: P(Doença|Teste+)
P_doenca_dado_positivo = (P_positivo_dado_doenca * P_doenca) / P_positivo

print(f'Probabilidade de doença dado teste positivo: {P_doenca_dado_positivo:.2%}')
print(f'\n⚠️ Mesmo com teste 99% preciso, a probabilidade real é apenas '
      f'{P_doenca_dado_positivo:.1%} porque a doença é rara!')

## 2️⃣ Entropia de Shannon

Entropia mede a **incerteza** de uma distribuição.

```
H(X) = -Σ p(x) · log₂(p(x))
```

- Moeda justa (50/50): H = 1 bit → máxima incerteza
- Moeda viciada (99/1): H ≈ 0.08 bits → quase certeza
- Evento certo (100/0): H = 0 → sem incerteza

In [ ]:
def entropia(probs):
    """Calcula entropia de Shannon (base 2) de uma distribuição."""
    probs = np.array(probs)
    probs = probs[probs > 0]  # Evitar log(0)
    return -np.sum(probs * np.log2(probs))

# Exemplos
print('Entropia de diferentes distribuições:')
print(f'  Moeda justa [0.5, 0.5]:     {entropia([0.5, 0.5]):.4f} bits')
print(f'  Moeda viciada [0.9, 0.1]:   {entropia([0.9, 0.1]):.4f} bits')
print(f'  Dado justo (6 faces):        {entropia([1/6]*6):.4f} bits')
print(f'  Certeza [1.0, 0.0]:          {entropia([1.0, 0.0]):.4f} bits')

## 3️⃣ Visualizando Entropia vs Probabilidade

Para uma moeda com P(cara) = p e P(coroa) = 1-p:

In [ ]:
p_range = np.linspace(0.001, 0.999, 200)
H = -p_range * np.log2(p_range) - (1-p_range) * np.log2(1-p_range)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(p_range, H, 'b-', linewidth=2.5)
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Máxima incerteza')
ax.fill_between(p_range, H, alpha=0.1, color='blue')
ax.set_xlabel('P(cara)', fontsize=13)
ax.set_ylabel('Entropia (bits)', fontsize=13)
ax.set_title('Entropia Binária — Máxima em p=0.5', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.show()

## 4️⃣ Cross-Entropy: A Loss Function de Classificação

Em redes neurais, usamos **Cross-Entropy** para medir o quão diferente a previsão é da realidade:

```
CE(y, ŷ) = -Σ y · log(ŷ)
```

Para classificação binária (Binary Cross-Entropy):
```
BCE = -[y·log(ŷ) + (1-y)·log(1-ŷ)]
```

**Intuição:** Se o modelo diz 0.9 e a resposta é 1.0 → loss baixa. Se diz 0.1 e a resposta é 1.0 → loss altíssima.

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """Calcula BCE para um par (y_true, y_pred)."""
    eps = 1e-15  # Evitar log(0)
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Demonstração: quando y_true = 1
preds = np.linspace(0.01, 0.99, 100)
losses = [binary_cross_entropy(1.0, p) for p in preds]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(preds, losses, 'r-', linewidth=2.5)
ax.set_xlabel('Predição do modelo (ŷ)', fontsize=13)
ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=13)
ax.set_title('BCE quando y=1: quanto mais longe de 1, maior a punição', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.annotate('Modelo acerta!', xy=(0.9, 0.1), fontsize=12, color='green', fontweight='bold')
ax.annotate('Modelo erra feio!', xy=(0.1, 2.0), fontsize=12, color='red', fontweight='bold')
plt.show()

## 5️⃣ Entropia em Árvores de Decisão

Árvores de decisão escolhem splits que **maximizam a redução de entropia** (Information Gain):
```
IG = H(pai) - Σ (Nᵢ/N) · H(filhoᵢ)
```
O split que mais reduz a incerteza é escolhido.

In [ ]:
# Simulação: split de dados binários
# Dataset: 10 amostras, 6 positivas, 4 negativas
total = np.array([6, 4])  # [positivos, negativos]
H_pai = entropia(total / total.sum())
print(f'Entropia do nó pai: {H_pai:.4f} bits')

# Split A: filho esquerdo [5,1], filho direito [1,3]
filho_e = np.array([5, 1])
filho_d = np.array([1, 3])
H_split_A = (filho_e.sum()/total.sum() * entropia(filho_e/filho_e.sum()) +
             filho_d.sum()/total.sum() * entropia(filho_d/filho_d.sum()))
IG_A = H_pai - H_split_A

# Split B: filho esquerdo [3,3], filho direito [3,1]
filho_e2 = np.array([3, 3])
filho_d2 = np.array([3, 1])
H_split_B = (filho_e2.sum()/total.sum() * entropia(filho_e2/filho_e2.sum()) +
             filho_d2.sum()/total.sum() * entropia(filho_d2/filho_d2.sum()))
IG_B = H_pai - H_split_B

print(f'\nSplit A: IG = {IG_A:.4f} bits')
print(f'Split B: IG = {IG_B:.4f} bits')
print(f'\n→ A árvore escolheria o Split {"A" if IG_A > IG_B else "B"} (maior Information Gain)')

## 🏋️ Questões para Praticar

### Q1. Bayes Invertido
No exemplo médico, se a prevalência sobe para 10%, qual é P(doença|teste+)? Compare com o resultado de 1%.

### Q2. Entropia Multiclasse
Calcule a entropia de um classificador que prevê 3 classes com [0.7, 0.2, 0.1]. Compare com [0.33, 0.33, 0.34].

### Q3. Implementar Cross-Entropy Multiclasse
Implemente `CE(y, ŷ) = -Σ yᵢ·log(ŷᵢ)` para one-hot encoding. Teste com `y=[0,1,0]` e `ŷ=[0.1, 0.7, 0.2]`.

### Q4. KL Divergence
Pesquise e implemente a divergência KL entre duas distribuições: `KL(P||Q) = Σ P(x)·log(P(x)/Q(x))`. Teste com P=[0.5,0.5] e Q=[0.9,0.1].

In [ ]:
# ============================================
# Resolva as questões aqui
# ============================================

# Q1: Bayes com prevalência 10%


# Q2: Entropia multiclasse


# Q3: Cross-Entropy multiclasse


# Q4: KL Divergence
